In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load CSV file
data = pd.read_csv('/content/drive/MyDrive/upscaling/species-based/Others_training.csv')  # file path


In [ ]:
# Identify and remove rows with high volume variance
volume_mean = data.iloc[:, 2].mean()
volume_std = data.iloc[:, 2].std()
threshold = volume_mean + 3 * volume_std  # Consider values beyond 3 standard deviations as outliers
outliers = data[data.iloc[:, 2] > threshold]
data = data[data.iloc[:, 2] <= threshold]

# Report removed outliers
print("Removed Outlier Values:")
print(outliers.iloc[:, 2].values)

In [ ]:
rs = 22  # random_state

In [ ]:
# Training RF model considering all features

# Extract relevant columns
X = data.iloc[:, 5:]  # Features start from the sixth column
y = data.iloc[:, 2]    # Volume is in the third column

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.125, random_state=rs)

# Train Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=rs)
rf.fit(X_train, y_train)

# Compute variable importance
importances = rf.feature_importances_
feature_names = X.columns


# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_names, importances, color='skyblue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance in Random Forest')
plt.show()

# Predict on test data
y_pred = rf.predict(X_test)

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)/1000

print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'R-squared: {r2:.4f}')


In [ ]:
# Filtering out features having less than 95% importance

# Train the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=rs)
rf.fit(X_train, y_train)

# Compute feature importance
importances = rf.feature_importances_
feature_names = X.columns

# Sort features by importance
sorted_indices = np.argsort(importances)[::-1]
sorted_features = feature_names[sorted_indices]
sorted_importances = importances[sorted_indices]

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(sorted_features, sorted_importances, color='skyblue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance in Random Forest')
plt.gca().invert_yaxis()  # Highest importance at top
plt.show()

# Selecting the top k features
cumulative_importance = np.cumsum(sorted_importances)
threshold = 0.95  # Keep 95% of total importance
k = np.argmax(cumulative_importance >= threshold) + 1  # Smallest number of features reaching threshold

# Reduce dataset to the top k important features
selected_features = sorted_features[:k]
X_reduced_o = X[selected_features]

print(f"Optimal number of features: {k}")
print(f"Selected Features: {list(selected_features)}")

# Train a new model using reduced features
X_train, X_test, y_train, y_test = train_test_split(X_reduced_o, y, test_size=0.125, random_state=rs)
rf_reduced = RandomForestRegressor(n_estimators=100, random_state=rs)
rf_reduced.fit(X_train, y_train)
y_pred = rf_reduced.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)/1000

print(f'Root Mean Squared Error with reduced features: {rmse:.4f}')
print(f'R-squared with reduced features: {r2:.4f}')


In [ ]:
# Removing redundant features

import seaborn as sns

# Compute correlation matrix
corr_matrix = X_reduced_o.corr().abs()

# Find features with high correlation
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]

# Remove redundant features
X_reduced_1 = X_reduced_o.drop(columns=redundant_features)

print(f"Removed {len(redundant_features)} redundant features.")
print(f"Final selected features: {list(X_reduced_1.columns)}")

# Train a new model using reduced features
X_train, X_test, y_train, y_test = train_test_split(X_reduced_1, y, test_size=0.125, random_state=rs)
rf_reduced = RandomForestRegressor(n_estimators=100, random_state=rs)
rf_reduced.fit(X_train, y_train)
y_pred = rf_reduced.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)/1000

print(f'Root Mean Squared Error with reduced features: {rmse:.4f}')
print(f'R-squared with reduced features: {r2:.4f}')


In [ ]:
from sklearn.feature_selection import RFECV

# Define a model
rf = RandomForestRegressor(n_estimators=100, random_state=rs)

# Perform RFECV
rfecv = RFECV(estimator=rf, step=1, cv=10, scoring='r2')  # 10-fold CV
X_reduced_2 = rfecv.fit_transform(X_reduced_1, y)  # Select important features

# Get the optimal number of features
optimal_features = X_reduced_1.columns[rfecv.support_]
print(f"Optimal Number of Features: {rfecv.n_features_}")
print(f"Selected Features: {list(optimal_features)}")


# Train and Evaluate Model with Selected Features
X_train, X_test, y_train, y_test = train_test_split(X_reduced_2, y, test_size=0.125, random_state=rs)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Compute Performance Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)/1000

print(f'Root Mean Squared Error with RFECV: {rmse:.4f}')
print(f'R-squared with RFECV: {r2:.4f}')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert X_reduced_2 back to DataFrame with column names
X_reduced_df = pd.DataFrame(X_reduced_2, columns=selected_features)

# Compute correlation matrix
corr_matrix = X_reduced_df.corr().abs()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()



In [ ]:
# Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.9
redundant_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]

# Remove redundant features from X_reduced_df
X_final = X_reduced_df.drop(columns=redundant_features)

print(f"Removed {len(redundant_features)} redundant features.")
print(f"Final selected features: {list(X_final.columns)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.125, random_state=rs)

# Train the model
rf_final = RandomForestRegressor(n_estimators=100, random_state=rs)
rf_final.fit(X_train, y_train)
y_pred = rf_final.predict(X_test)

# Compute Performance Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)/1000

print(f'Final Root Mean Squared Error: {rmse:.4f}')
print(f'Final R-squared: {r2:.4f}')


In [ ]:
# Scatter plot with fitted line
plt.figure(figsize=(8, 6))
plt.scatter(y_test_rfe, y_pred_rfe, color='blue', alpha=0.6, label='Predicted vs Actual')
plt.plot([min(y_test_rfe), max(y_test_rfe)], [min(y_test_rfe), max(y_test_rfe)], color='red', linestyle='dashed', label='1:1 line')
plt.xlabel('Actual Volume')
plt.ylabel('Predicted Volume')
plt.title(f'Scatter Plot of Testing Dataset (R² = {r2_rfe:.4f})')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

# Load CSV file
file_path = "/content/drive/MyDrive/upscaling/validation_results.csv"  # Change to your actual file path
data = pd.read_csv(file_path)

# Extract reference and predicted volumes (assuming 3rd & 4th columns)
y_true = data.iloc[:, 2]  # Reference Volume (3rd column)
y_pred = data.iloc[:, 3]  # Predicted Volume (4th column)

# Compute R² and RMSE
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Print results
print(f'R-squared (R²): {r2:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')

# Plot scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_true, y_pred, color='blue', alpha=0.6, edgecolors='k')
plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', lw=2)  # 1:1 line
plt.xlabel("Reference Volume")
plt.ylabel("Predicted Volume")
plt.title("Scatter Plot of Reference vs. Predicted Volume")
plt.grid(True)
plt.show()


In [ ]:
# Save predictions of test dataset

column_ids = ['TreeID', 'CentroidX', 'CentroidY']

# Convert X_test back to DataFrame (if it's an array)
X_test_df = pd.DataFrame(X_test, columns=selected_features)

# Extract treeid, x, and y from the original dataset (assuming X_lasso_df has them)
treeid_x_y = X_test_df

# Combine with actual and predicted volume
results_df = pd.concat([treeid_x_y.reset_index(drop=True), X_test_df.reset_index(drop=True)], axis=1)
results_df['Actual_Volume'] = y_test.values
results_df['Predicted_Volume'] = y_pred

# Save to CSV
results_df.to_csv("/content/drive/MyDrive/upscaling/RF_predictions_test.csv", index=False)

print("Test results saved to 'gradient_boosting_test_results.csv'")
